<a href="https://colab.research.google.com/github/Ahowardjr/MSSP6070/blob/main/WeeklyModules/Week%2010/Week_10_Assignment(Resubmission_of_Week_4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pathlib import Path
import pandas as pd

# Load the earthquake dataset.
file_path = Path("/content/parsed.csv")

if not file_path.is_file():
    raise FileNotFoundError(
        "Upload parsed.csv to Colab using the folder icon on the left."
    )

df = pd.read_csv(file_path)

# Check that the required columns are available.
required_columns = ["place", "magType", "mag", "tsunami"]
missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Check numeric values without silently removing invalid entries.
for column in ["mag", "tsunami"]:
    df[column] = pd.to_numeric(df[column], errors="raise")

if not df["tsunami"].dropna().isin([0, 1]).all():
    raise ValueError("Expected tsunami values of 0, 1, or missing.")

print(f"Loaded {len(df):,} earthquake records.")
print("\nMissing values in required columns:")
print(df[required_columns].isna().sum())


# A. Find the 95th percentile of Japan earthquakes with magType "mb".
# na=False treats missing locations as nonmatches.
# regex=False searches for literal text.
japan = df[
    df["place"].str.contains("Japan", na=False, regex=False)
]

japan_mb = japan[japan["magType"] == "mb"]

if japan_mb["mag"].notna().any():
    japan_95th = japan_mb["mag"].quantile(0.95)
    print(f"\nA. Japan mb magnitude, 95th percentile: {japan_95th:.2f}")
else:
    print("\nA. No valid Japan mb magnitudes available.")


# B. Calculate the percentage of Indonesia records flagged for tsunami.
indonesia = df[
    df["place"].str.contains("Indonesia", na=False, regex=False)
]

indonesia_tsunamis = indonesia[indonesia["tsunami"] == 1]

if len(indonesia) > 0:
    percentage = len(indonesia_tsunamis) / len(indonesia) * 100
    print(f"\nB. Indonesia records flagged for tsunami: {percentage:.3f}%")
    print(f"Missing tsunami flags: {indonesia['tsunami'].isna().sum()}")
else:
    print("\nB. No Indonesia records found; percentage cannot be calculated.")

# This uses all Indonesia records as the denominator, as in the original code.


# C. Display summary statistics for earthquakes in Nevada.
nevada = df[
    df["place"].str.contains("Nevada", na=False, regex=False)
]

print("\nC. Nevada summary statistics:")
display(nevada.describe())


# D. Identify earthquakes in the assignment's Ring of Fire locations.
ring_locations = [
    "Bolivia", "Chile", "Ecuador", "Peru",
    "Costa Rica", "Guatemala", "Mexico",
    "Japan", "Philippines", "Indonesia",
    "New Zealand", "Antarctic", "Canada",
    "Fiji", "Alaska", "Washington", "California",
    "Russia", "Taiwan", "Tonga", "Kermadec Islands"
]

ring_results = []

for place in df["place"].fillna(""):
    place_lower = place.lower()
    on_ring = False

    for location in ring_locations:
        if location.lower() in place_lower:

            # Do not count New Mexico as Mexico.
            if location == "Mexico" and "new mexico" in place_lower:
                continue

            on_ring = True
            break

    ring_results.append(on_ring)

# Add the indicator to a copy so df does not receive the new column.
analysis_df = df.copy()
analysis_df["ring_of_fire"] = ring_results

print("\nD. Sample Ring of Fire results:")
display(analysis_df[["place", "ring_of_fire"]].head())


# E. Count earthquakes inside and outside the listed locations.
ring_counts = analysis_df["ring_of_fire"].value_counts()

print(f"\nE. Inside listed locations: {ring_counts.get(True, 0):,}")
print(f"Outside listed locations: {ring_counts.get(False, 0):,}")
print(f"Missing locations: {analysis_df['place'].isna().sum():,}")


# F. Count tsunami-flagged earthquakes in the Ring of Fire locations.
ring_tsunamis = analysis_df[
    analysis_df["ring_of_fire"]
    & (analysis_df["tsunami"] == 1)
]

print(
    "\nF. Tsunami-flagged records in Ring of Fire locations: "
    f"{len(ring_tsunamis):,}"
)

Loaded 9,332 earthquake records.

Missing values in required columns:
place      0
magType    1
mag        1
tsunami    0
dtype: int64

A. Japan mb magnitude, 95th percentile: 4.90

B. Indonesia records flagged for tsunami: 23.129%
Missing tsunami flags: 0

C. Nevada summary statistics:


,cdi,dmin,felt,gap,mag,mmi,nst,rms,sig,time,tsunami,tz,updated
count,15.000000,677.000000,15.000000,677.000000,677.000000,1.00,677.000000,677.000000,677.000000,6.770000e+02,677.0,677.0,6.770000e+02
mean,2.440000,0.166982,2.400000,154.029527,0.491728,2.84,12.608567,0.151909,10.688331,1.538314e+12,0.0,-480.0,1.538402e+12
std,0.501142,0.166400,4.626013,68.769713,0.689560,NaN,9.890620,0.084742,19.252727,5.954070e+08,0.0,0.0,6.000267e+08
min,2.000000,0.001000,1.000000,29.140000,-0.500000,2.84,3.000000,0.000500,0.000000,1.537247e+12,0.0,-480.0,1.537307e+12
25%,2.000000,0.054000,1.000000,97.670000,-0.100000,2.84,6.000000,0.106900,0.000000,1.537854e+12,0.0,-480.0,1.537928e+12
50%,2.200000,0.113000,1.000000,149.550000,0.400000,2.84,9.000000,0.146300,2.000000,1.538280e+12,0.0,-480.0,1.538428e+12
75%,2.900000,0.234000,1.000000,200.470000,0.900000,2.84,16.000000,0.186700,12.000000,1.538821e+12,0.0,-480.0,1.538878e+12
max,3.300000,1.414000,19.000000,355.910000,2.900000,2.84,61.000000,0.863400,129.000000,1.539461e+12,0.0,-480.0,1.539483e+12



D. Sample Ring of Fire results:


,place,ring_of_fire
0,"9km NE of Aguanga, CA",False
1,"9km NE of Aguanga, CA",False
2,"8km NE of Aguanga, CA",False
3,"9km NE of Aguanga, CA",False
4,"10km NW of Avenal, CA",False



E. Inside listed locations: 4,426
Outside listed locations: 4,906
Missing locations: 0

F. Tsunami-flagged records in Ring of Fire locations: 43
